## Reclaim policies & the CSI

### Reclaim policies — what happens when a PVC is deleted

The bound PV's **reclaim policy** decides the disk's fate:

| Policy | Meaning |
|---|---|
| `Delete` | Delete the disk *and* the PV object. **Data is gone.** Default for dynamic PVs. |
| `Retain` | Keep both. The PV goes `Released` (not `Available`); an admin must clean up before another PVC can bind. For curated, valuable storage. |
| `Recycle` | *Deprecated* — use dynamic provisioning instead. |

Dynamic PVs inherit the policy from the StorageClass (usually `Delete`); static PVs set it directly. Worth switching your prod-database PV to `Retain` even if the class says `Delete` — defence against an accidental PVC deletion:

```bash
kubectl patch pv <pv> -p '{"spec":{"persistentVolumeReclaimPolicy":"Retain"}}'
```

A `Retain`ed PV stays `Released` (leftover data + a `claimRef`) until an admin removes the `claimRef`.

### The Container Storage Interface

Kubernetes doesn't know how to create an EBS disk or mount NFS — **CSI drivers** do, via a standard gRPC interface. Each driver is two parts: a **controller** Deployment (talks to the backend API — create/delete/snapshot/expand) and a **node** DaemonSet (attaches + mounts into the pod). Every backend speaks the same CSI, so Kubernetes stays storage-agnostic.

You rarely write CSI. For the CKA: drivers are Helm-installed and referenced by *provisioner name* in a StorageClass; `kubectl get csidrivers` / `csinodes` list them; snapshots, clones, and expansion are all CSI features the driver opts into. Older in-tree drivers are deprecated in favour of these out-of-tree pods. On our map this is the **CSI** chip — the driver beneath the StorageClass that does the physical work.